[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_7/L3.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 7 | BÀI 3


---

# **ĐÁNH GIÁ MÔ HÌNH VÀ ĐIỀU CHUẨN HÓA**


|  |  |
|:---|:---|
|**Thời gian đọc** | 40 phút |
|**Kiến thức nền** | Giới thiệu về tinh chỉnh siêu tham số, hàm mất mát  |
|**Từ khóa** |luyện kim mô phỏng (simulated annealing)  |


---

*Ở bài trước, ta đã áp dụng các kỹ thuật tinh chỉnh siêu tham số bằng tìm kiếm lưới và tìm kiếm ngẫu nhiên vào bài toán dự báo giá nhà và đánh giá hiệu suất mô hình. Trong bài này, ta sẽ trình bày những hiểu biết về thời điểm nên dùng các chiến lược kiểm định chéo khác nhau. Ta cũng sẽ bàn về các thước đo đánh giá mô hình và giới thiệu khái niệm có một mô hình cơ sở để so sánh với hiệu suất mô hình của ta.*

## **1. Đánh giá mô hình**

Ở các module trước, ta đã gặp nhiều thuật toán học khác nhau, và ở tất cả chúng, ta đều quan tâm đến việc cực tiểu hóa sai số khi gặp dữ liệu chưa từng thấy. Để đạt được điều này, ta sẽ bàn về các khái niệm sau trong phần này:
1. Chọn chiến lược kiểm định chéo phù hợp.
2. Xây dựng mô hình cơ sở đóng vai trò tham chiếu cho mô hình thực.
3. Các khái niệm kiểm định chéo lồng nhau và đánh giá của chúng.
4. Các thước đo dùng trong mô hình phân loại và hồi quy.

### **1.1 Xây dựng mô hình cơ sở**

Mô hình cơ sở (baseline) có thể xem là mô hình tham chiếu cho mô hình thực. Ta nên cân nhắc các điểm sau khi xây dựng mô hình cơ sở:
1. Nó nên là mô hình đơn giản dùng để so sánh.
2. Nó nên dễ giải thích.
3. Nó nên dựa trên dữ liệu thực.

Mô hình cơ sở quan trọng vì các lý do sau;

**Hiểu dữ liệu**

Khi huấn luyện một mô hình cơ sở trên tập dữ liệu, ta có thể suy ra:
- Một mô hình cơ sở mà dự báo biến mục tiêu của ta hoàn toàn sai cho thấy các mô hình yếu hơn sẽ khó nắm bắt các mẫu hình trong tập dữ liệu đến mức nào.
- Nó có thể giúp ta chỉ ra phân đoạn dữ liệu làm mô hình có sai số cao hơn.

**Mô hình hóa nhanh hơn**

Khi đã có mô hình cơ sở, việc xây dựng quy trình mô hình thực trở nên dễ hơn vì ta hiểu dữ liệu tốt hơn.

**Chuẩn đối chiếu hiệu suất**

Hiệu suất của mô hình cơ sở giúp ta quyết định cần mô hình phức tạp hay đơn giản.

Ta có thể chia mô hình cơ sở thành hai loại:
1. Mô hình cơ sở đơn giản.
2. Mô hình cơ sở học máy.

Trong mục này, ta sẽ xét một mô hình cơ sở đơn giản dùng logic đơn giản để xây dựng mô hình cơ sở.

In [ ]:
from sklearn.datasets import fetch_california_housing

data, target = fetch_california_housing(return_X_y=True, as_frame=True)

Ta chia dữ liệu bằng `ShuffleSplit` với $20\%$ dữ liệu dùng để kiểm định.

In [ ]:
from sklearn.model_selection import ShuffleSplit

cv = ShuffleSplit(n_splits=30, test_size=0.2, random_state=0)

Ta định nghĩa một hàm hồi quy cây quyết định làm mô hình cho phần này.

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor

regressor = DecisionTreeRegressor()
cv_results_tree_regressor = cross_validate(
    regressor, data, target, cv=cv, scoring="neg_mean_absolute_error", n_jobs=2
)

errors_tree_regressor = pd.Series(
    -cv_results_tree_regressor["test_score"], name="Decision tree regressor"
)

Ta dùng bộ hồi quy giả (dummy regressor) làm mô hình cơ sở, dự báo bằng trung bình của mục tiêu đã tính.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
result_dummy = cross_validate(
    dummy, data, target, cv=cv, scoring="neg_mean_absolute_error", n_jobs=2
)
errors_dummy_regressor = pd.Series(-result_dummy["test_score"], name="Dummy regressor")
errors_dummy_regressor.describe()

Trong ô tiếp theo, ta chuẩn bị một bảng so sánh hiệu suất của bộ hồi quy cây quyết định và bộ hồi quy giả.

In [ ]:
all_errors = pd.concat(
    [errors_tree_regressor, errors_dummy_regressor],
    axis=1,
)
all_errors

Từ bảng trên ta thấy ở mọi lần chia, bộ hồi quy cây quyết định có sai số ít hơn so với bộ hồi quy giả. Trong ô tiếp theo, ta có thể trực quan hóa phân phối sai số của cả hai hàm hồi quy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bins = np.linspace(start=0, stop=1, num=80)
all_errors.plot.hist(bins=bins, edgecolor="black")
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.xlabel("Mean absolute error")
plt.suptitle(
    "Fig. 1: Cross-validation testing errors.",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

### **1.2 Chọn chiến lược kiểm định chéo**

Đến giờ ta chỉ dùng `ShuffleSplit` hoặc `KFold` để kiểm định chéo, nhưng chúng không phải lúc nào cũng hiệu quả. Do đó, điều quan trọng là chọn chiến lược kiểm định chéo một cách khôn ngoan, tùy theo nhiệm vụ. Ta xét ví dụ bên dưới để minh họa vì sao cần chọn cẩn thận chiến lược kiểm định chéo.

In [ ]:
from sklearn.datasets import load_iris

data, target = load_iris(as_frame=True, return_X_y=True)

Bây giờ ta tạo một thuật toán phân loại đơn giản cho nhiệm vụ này.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression())

Bây giờ ta dùng mô hình này để huấn luyện và khái quát hóa trên dữ liệu chưa thấy.

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=3)
results = cross_validate(model, data, target, cv=cv)
test_score = results["test_score"]
print(f"The average accuracy is " f"{test_score.mean():.3f} ± {test_score.std():.3f}")

Kết quả trên cho thấy mô hình không dự báo đúng được cái nào khi gặp dữ liệu kiểm định. Để hiểu kết quả kỳ lạ này, ta xem xét đặc trưng mục tiêu.

In [ ]:
import matplotlib.pyplot as plt

target.plot()
plt.xlabel("Sample index")
plt.ylabel("Class")
plt.yticks(target.unique())
plt.suptitle(
    "Fig. 2: Class value in target y.", fontweight="bold", horizontalalignment="right"
)
plt.show()

Ta có thể quan sát từ sơ đồ trên rằng cột mục tiêu được sắp xếp theo thứ tự, và điều này gây ra kết quả không mong đợi khi ta dùng kiểm định chéo. Bây giờ ta tính số đếm các nhóm cho tập huấn luyện và tập kiểm định trong các ô bên dưới.

In [ ]:
import pandas as pd

n_splits = 3
cv = KFold(n_splits=n_splits)

train_cv_counts = []
test_cv_counts = []
for fold_idx, (train_idx, test_idx) in enumerate(cv.split(data, target)):
    target_train, target_test = target.iloc[train_idx], target.iloc[test_idx]

    train_cv_counts.append(target_train.value_counts())
    test_cv_counts.append(target_test.value_counts())

Ta hiển thị điều này trong bảng bên dưới.

In [ ]:
train_cv_counts = pd.concat(
    train_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
train_cv_counts.index.name = "Class label"
train_cv_counts

Ta có thể thấy khi huấn luyện mô hình, ở mỗi phần chỉ có hai nhóm có sẵn, và do đó khó dự báo trên dữ liệu chưa thấy.

In [ ]:
test_cv_counts = pd.concat(
    test_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
test_cv_counts.index.name = "Class label"
test_cv_counts

Trong dữ liệu kiểm tra, chỉ dùng một nhóm, mà nhóm này không có trong huấn luyện. Điều này làm khó dự báo, vì nhóm đó không có khi huấn luyện. Các ô bên dưới giúp trực quan hóa các bảng trên.

In [ ]:
train_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 3: Training Set Data Distribution in Folds.",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

In [ ]:
test_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 4: Data distribution in Testing Set",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Để giải quyết các vấn đề trên, một thông lệ tốt là xáo trộn tập dữ liệu trước khi bắt đầu giai đoạn mô hình hóa như bên dưới.

In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=0)
results = cross_validate(model, data, target, cv=cv)
test_score = results["test_score"]
print(f"The average accuracy is " f"{test_score.mean():.3f} ± {test_score.std():.3f}")

Bây giờ ta thấy độ chính xác của mô hình thực sự đã cải thiện lên hơn $95\%$. Bây giờ ta có thể trực quan hóa số đếm các nhóm ở mỗi phần cho cả mẫu huấn luyện và mẫu kiểm tra.

In [ ]:
train_cv_counts = []
test_cv_counts = []
for fold_idx, (train_idx, test_idx) in enumerate(cv.split(data, target)):
    target_train, target_test = target.iloc[train_idx], target.iloc[test_idx]

    train_cv_counts.append(target_train.value_counts())
    test_cv_counts.append(target_test.value_counts())
train_cv_counts = pd.concat(
    train_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
test_cv_counts = pd.concat(
    test_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
train_cv_counts.index.name = "Class label"
test_cv_counts.index.name = "Class label"

In [ ]:
train_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 5: Training set Data Distribution (kFold).",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Từ các biểu đồ cột trên, ta có thể quan sát thấy các nhóm được phân bố khá đều ở mỗi phần. Điều này cũng đúng với tập dữ liệu kiểm tra bên dưới.

In [ ]:
test_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 6: Testing set Data Distribution (kFold).",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Một kỹ thuật khác là chia tập dữ liệu để giữ nguyên số đếm các nhóm ban đầu ở mỗi lần chia kiểm định. Điều này thực hiện bằng phân tầng như trong các ô bên dưới. Các bước vẫn như những gì ta đã thấy ở hai chiến lược trước.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=3)

In [ ]:
results = cross_validate(model, data, target, cv=cv)
test_score = results["test_score"]
print(f"The average accuracy is " f"{test_score.mean():.3f} ± {test_score.std():.3f}")

In [ ]:
train_cv_counts = []
test_cv_counts = []
for fold_idx, (train_idx, test_idx) in enumerate(cv.split(data, target)):
    target_train, target_test = target.iloc[train_idx], target.iloc[test_idx]

    train_cv_counts.append(target_train.value_counts())
    test_cv_counts.append(target_test.value_counts())
train_cv_counts = pd.concat(
    train_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
test_cv_counts = pd.concat(
    test_cv_counts, axis=1, keys=[f"Fold #{idx}" for idx in range(n_splits)]
)
train_cv_counts.index.name = "Class label"
test_cv_counts.index.name = "Class label"

In [ ]:
train_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 7: Training set Data Distribution (Stratified).",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

In [ ]:
test_cv_counts.plot.bar()
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
plt.ylabel("Count")
plt.suptitle(
    "Fig. 8: Testing set Data Distribution (Stratified).",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Ta có thể thấy số đếm ở tập huấn luyện và tập kiểm tra không quá xa nhau.

### **1.3 Kiểm định chéo lồng nhau**

Kỹ thuật kiểm định chéo k lần mà ta đã bàn đến giờ rất quan trọng cho việc tối ưu siêu tham số và đánh giá mô hình trên tập kiểm định. Việc ta dùng kiểm định chéo trên cùng một tập dữ liệu để vừa tinh chỉnh siêu tham số vừa chọn mô hình từ cùng quá trình đó có thể làm việc đánh giá mô hình bị chệch.

Một cách khắc phục hạn chế này là tạo một vòng ngoài của kiểm định chéo. Quá trình này gọi là **kiểm định chéo lồng nhau** (nested cross-validation) và ta sẽ minh họa trong các ô bên dưới.

Sau đó ta cấu hình quy trình kiểm định chéo bên dưới. Ta sẽ dùng tập dữ liệu make classification.

In [ ]:
from numpy import mean, std
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

Sau đó ta chia tập dữ liệu thành tập huấn luyện và tập kiểm tra, trong đó tập huấn luyện cũng sẽ được dùng để huấn luyện và kiểm định mô hình.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data, target, shuffle=True, random_state=0, test_size=0.3
)

Sau đó ta định nghĩa mô hình dùng cho bài toán này.

In [ ]:
model = RandomForestClassifier(random_state=1)

Bây giờ ta cấu hình vòng kiểm định chéo trong, dùng để huấn luyện các siêu tham số.

In [ ]:
cv_inner = KFold(n_splits=3, shuffle=True, random_state=1)

Như đã thấy ở Bài 1, ta định nghĩa không gian tìm kiếm cho các siêu tham số cần tinh chỉnh và áp dụng tìm kiếm lưới để có tổ hợp siêu tham số tốt nhất.

In [ ]:
space = dict()
space["n_estimators"] = [10, 100, 500]
space["max_features"] = [2, 4, 6]
# Search for the best hyperparameter combination
search = GridSearchCV(
    model, space, scoring="accuracy", n_jobs=1, cv=cv_inner, refit=True
)

Bây giờ ta cấu hình vòng kiểm định chéo giúp ta đánh giá mô hình.

In [ ]:
cv_outer = KFold(n_splits=10, shuffle=True, random_state=1)

Ô tiếp theo thực hiện kiểm định chéo lồng nhau trên dữ liệu và ta xuất kết quả.

In [ ]:
# execute the nested cross-validation
scores = cross_val_score(
    search, X_train, y_train, scoring="accuracy", cv=cv_outer, n_jobs=-1
)
# report performance
print("Accuracy: %.3f (%.3f)" % (mean(scores), std(scores)))

Ta có độ chính xác trung bình của mô hình từ 10 vòng lặp, và kết quả cho thấy mô hình của ta có độ chính xác tốt hơn. Nhưng trong phân loại, ta cần đi xa hơn độ chính xác như một thước đo hiệu suất mô hình. Ở phần tiếp theo, ta học về các thước đo phân loại.

## **2. Các thước đo trong học máy**

Khi xây dựng các mô hình học máy, ta cần đo hiệu suất của chúng, và hiểu loại thước đo cần dùng là điều thiết yếu để ta đánh giá liệu mình có tiến bộ hay không. Như bây giờ ta đều biết, các nhiệm vụ học máy có thể chia thành hồi quy hoặc phân loại, và chúng được xử lý khác nhau khi mô hình hóa và cả khi đo hiệu suất. Ta sẽ bắt đầu bằng việc xem các thước đo hồi quy.

### **2.1 Các thước đo hồi quy**

Hồi quy tuyến tính gồm việc tìm mối quan hệ tuyến tính tồn tại giữa các biến độc lập và biến mục tiêu, tức là
$$y_i (x_i, w) = w_0 + w_1 x_{i1} + w_2 x_{i2} + \cdots w_m x_{im} + ᵋ_i$$
trong đó $y\in \mathbb{R}$ là biến mục tiêu, $x \in \mathbb{R}^m$ là các biến dự báo và $ᵋ_i$ là sai số ngẫu nhiên. Sai số ngẫu nhiên xảy ra do một trong các nguyên nhân sau
- Chọn sai mô hình tuyến tính.
- Chọn các biến ít liên quan và bỏ sót các biến liên quan.
- Sai số đo lường.

Một hồi quy tuyến tính tốt ước lượng hệ số của các đặc trưng, $w_i$, và cực tiểu hóa các sai số phần dư, $e_i = y_i  - \hat{y}_i$. Điều này khả thi bằng cách cực tiểu hóa hàm mục tiêu. Hàm mục tiêu có thể nhận một trong các dạng bên dưới.
 

 #### **2.1.1 Sai số bình phương trung bình (MSE)**
 Sai số bình phương trung bình cho bởi $$L = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2$$

 #### **2.1.2 Sai số tuyệt đối trung bình (MAE)**

 Sai số tuyệt đối trung bình cho bởi
 $$L = \frac{1}{n} \sum_{i=1}^n |\hat{y}_i - y_i|$$

 #### **2.1.3 Sai số bình phương logarit trung bình**

 Thước đo này tương tự MSE, nhưng biến phụ thuộc ở thang logarit:
 $$L = \frac{1}{n} \sum_{i=1}^n (\log(\hat{y}_i+1) - log(y_i+1))^2$$

Trong các ô tiếp theo, ta sẽ xem các hàm mục tiêu khác nhau trong hồi quy để đo sai số.

In [ ]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data, target, shuffle=True, random_state=0
)

Trước hết ta huấn luyện dữ liệu bằng bộ hồi quy cây quyết định rồi đánh giá trên tập dữ liệu kiểm tra. Trong các ô tiếp theo, ta xem các thước đo khác nhau dùng để đánh giá mô hình học máy. Ta bắt đầu với thước đo dùng phổ biến nhất, là sai số bình phương trung bình.

In [ ]:
from sklearn.metrics import mean_squared_error

regressor.fit(data_train, target_train)
target_predicted = regressor.predict(data_train)

print(
    f"Mean squared error on the training set: "
    f"{mean_squared_error(target_train, target_predicted):.3f}"
)

Ta đo sai số bình phương trung bình trên dữ liệu kiểm tra bên dưới.

In [ ]:
target_predicted = regressor.predict(data_test)

print(
    f"Mean squared error on the testing set: "
    f"{mean_squared_error(target_test, target_predicted):.3f}"
)

Để tính hệ số xác định, $R^2$, ta tính hàm `score()` như bên dưới.

In [ ]:
regressor.score(data_test, target_test)

Ta cũng có thể tính sai số tuyệt đối trung bình cho mô hình như bên dưới.

In [ ]:
from sklearn.metrics import mean_absolute_error

target_predicted = regressor.predict(data_test)
print(
    f"Mean absolute error: " f"{mean_absolute_error(target_test, target_predicted):.3f}"
)

Sai số tuyệt đối trung vị cũng được tính tương tự.

In [ ]:
from sklearn.metrics import median_absolute_error

print(
    f"Median absolute error: "
    f"{median_absolute_error(target_test, target_predicted):.3f} "
)

#### **2.1.4 Điều chuẩn hóa**

Các kỹ thuật điều chuẩn hóa được dùng trong các trường hợp mô hình hồi quy quá khớp. Mục tiêu của điều chuẩn hóa là thêm một số hạng phạt vào hàm mục tiêu để điều chỉnh độ lớn của các tham số, tức là,

$$L = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2 + \lambda R(\vec{w})$$
trong đó $R(\vec{w})$ là số hạng phạt, một siêu tham số.

Các kỹ thuật điều chuẩn hóa dùng phổ biến nhất là hồi quy ridge và hồi quy lasso.

 **1. Hồi quy Ridge**

Hồi quy Ridge thêm một số hạng chuẩn $L2$ vào hàm mục tiêu. Số hạng này gây ra suy giảm trọng số cho các biến. Hàm mất mát của hồi quy ridge cho bởi
$$L = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2 + \frac{\lambda}{2}||w||^2$$

Siêu tham số $\lambda$ kiểm soát mức suy giảm, trong đó giá trị $\lambda$ lớn hơn làm tăng suy giảm và do đó làm mô hình vững hơn trước đa cộng tuyến. Lưu ý với giá trị $\lambda = 0$, hồi quy ridge tương đương hồi quy tuyến tính. Tăng giá trị $\lambda$ làm các hệ số của mô hình tiến về không.

Ta bắt đầu bằng việc nhập các thư viện cho phần minh họa này.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_squared_error

sns.set_style("darkgrid")

Ta nhập và chia tập dữ liệu thành các tập huấn luyện và kiểm tra.

In [ ]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X.head()

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25
)

Ta xây dựng mô hình ridge bên dưới và khớp nó vào tập huấn luyện.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

steps = [
    ("scalar", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2)),
    ("model", Ridge()),
]

ridge_pipe = Pipeline(steps)
ridge_pipe.fit(X_train, y_train)
# Predicting the Test set results
y_pred = ridge_pipe.predict(X_test)

Ta áp dụng kiểm định chéo k lần trên tập dữ liệu.

In [ ]:
from sklearn.model_selection import cross_val_score

accuracies = cross_val_score(estimator=ridge_pipe, X=X_train, y=y_train, cv=10)
print("accuracy mean", accuracies.mean())
print("accuracy standard deviation", accuracies.std())

Ta áp dụng tìm kiếm lưới để tìm mô hình tốt nhất và tham số alpha tốt nhất.

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

parameters = [{"model__alpha": np.arange(0, 0.2, 0.01)}]

scoring_func = make_scorer(mean_squared_error)
grid_search = GridSearchCV(
    estimator=ridge_pipe, param_grid=parameters, scoring=scoring_func, cv=10, n_jobs=-1
)
grid_search = grid_search.fit(X_train, y_train)

Trong ô bên dưới, ta hiển thị các giá trị alpha mà ta định tinh chỉnh.

In [ ]:
grid_search.cv_results_["params"]

Như đã bàn ở phần đánh giá mô hình, ta sẽ đánh giá hiệu suất của hồi quy ridge trên tập dữ liệu.

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RepeatedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

Ta áp dụng hồi quy tuyến tính trên tập dữ liệu và tạo dự báo trên tập kiểm tra.

In [ ]:
lreg = LinearRegression()
lreg.fit(X_train, y_train)

lreg_y_pred = lreg.predict(X_test)

# calculating Mean Squared Error (MSE)
mean_squared_error = np.mean((lreg_y_pred - y_test) ** 2)
print("Mean squared Error on test set : ", mean_squared_error)

# Putting together the coefficient and their corresponding variable names
lreg_coefficient = pd.DataFrame()
lreg_coefficient["Columns"] = X_train.columns
lreg_coefficient["Coefficient Estimate"] = pd.Series(lreg.coef_)
print(lreg_coefficient)

Trong ô tiếp theo, ta hiển thị các hệ số của đặc trưng.

In [ ]:
# plotting the coefficient score
fig, ax = plt.subplots(figsize=(20, 10))

ax.bar(
    lreg_coefficient["Columns"],
    lreg_coefficient["Coefficient Estimate"],
)

ax.spines["bottom"].set_position("zero")

plt.style.use("ggplot")
plt.suptitle(
    "Fig. 9: Coefficients Scores.", fontweight="bold", horizontalalignment="right"
)
plt.show()

In [ ]:
# import ridge regression from `sklearn` library
from sklearn.linear_model import Ridge

# Train the model
ridgeR = Ridge(alpha=1)
ridgeR.fit(X_train, y_train)
y_pred = ridgeR.predict(X_test)

# calculate mean square error
mean_squared_error_ridge = np.mean((y_pred - y_test) ** 2)
print(mean_squared_error_ridge)

# get ridge coefficient and print them
ridge_coefficient = pd.DataFrame()
ridge_coefficient["Columns"] = X_train.columns
ridge_coefficient["Coefficient Estimate"] = pd.Series(ridgeR.coef_)
print(ridge_coefficient)

**2. Hồi quy Lasso**

Lasso (Least Absolute Shrinkage and Selection Operator) là một mô hình điều chuẩn hóa khác, thêm một số hạng $L1$ vào hàm mục tiêu. Nó hoạt động bằng cách cho các tham số không liên quan bằng không. Hàm mất mát của hồi quy lasso cho bởi

$$L = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2 + \frac{\lambda}{2}||w||$$

Như ở hồi quy ridge, $\lambda$ kiểm soát độ mạnh của mức phạt. Như hồi quy ridge ở trên, khi $\lambda = 0$, ta có phương trình hồi quy tuyến tính. Khi giá trị $\lambda$ tăng, các đặc trưng không quan trọng bị co lại và một số sẽ bằng không, phương sai của mô hình giảm và độ chệch tăng.

Trong các ô bên dưới, ta minh họa cách áp dụng lasso trên tập dữ liệu.

In [ ]:
steps = [("scalar", StandardScaler()), ("model", Lasso())]

lasso_pipe = Pipeline(steps)

Ta áp dụng kiểm định chéo và in ra sai số tuyệt đối trung bình và giá trị alpha tốt nhất.

In [ ]:
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
model__alpha = np.linspace(0.1, 2, 21)
lasso = Lasso()
grid = dict()
grid["model__alpha"] = model__alpha
gscv = GridSearchCV(
    lasso_pipe, grid, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1
)
results = gscv.fit(X, y)
print("MAE: %.5f" % results.best_score_)
print("Config: %s" % results.best_params_)

Bây giờ ta có thể nhập các thư viện và tính thước đo mô hình bên dưới.

In [ ]:
from sklearn.linear_model import Lasso

# Train the model
lasso = Lasso(alpha=1)
lasso.fit(X_train, y_train)
y_pred1 = gscv.predict(X_test)

# Calculate Mean Squared Error
mean_squared_error = np.mean((y_pred1 - y_test) ** 2)
print("Mean squared error on test set", mean_squared_error)
lasso_coeff = pd.DataFrame()
lasso_coeff["Columns"] = X_train.columns
lasso_coeff["Coefficient Estimate"] = pd.Series(lasso.coef_)

print(lasso_coeff)

Từ kết quả trên, ta thấy các hệ số của đặc trưng không liên quan được làm tròn xuống không.

### **2.2 Các thước đo phân loại**

Trong các ô tiếp theo, ta sẽ bàn và minh họa các thước đo phân loại khác nhau trên tập dữ liệu iris.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data, target = load_iris(as_frame=True, return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, shuffle=True, random_state=0, test_size=0.3
)

model = RandomForestClassifier(random_state=1)
model.fit(X_train, y_train)
target_predicted = model.predict(X_test)

**Độ chính xác**

Ta dùng độ chính xác làm cơ sở. Độ chính xác đơn giản tìm tỷ lệ các nhóm ta phân loại đúng, tức là

In [ ]:
y_test == target_predicted

Để có độ chính xác của mô hình, ta lấy trung bình số lần ta dự báo đúng như bên dưới.

In [ ]:
import numpy as np

np.round(np.mean(y_test == target_predicted), 3)

Điều này được cài đặt trong `sklearn` bằng `accuracy_score` như bên dưới.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, target_predicted)
print(f"Accuracy: {accuracy:.3f}")

**Ma trận nhầm lẫn**

Độ chính xác có thể không phải thước đo tốt cho hiệu suất mô hình, nhất là khi biến mục tiêu mất cân bằng. Nhóm phổ biến nhất sẽ luôn có trọng số lớn hơn. Để có chi tiết tinh hơn về hiệu suất mô hình, nên dựng ma trận nhầm lẫn để xác định độc lập sai số của từng nhóm. Ô bên dưới cho thấy cách dựng ma trận nhầm lẫn

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

_ = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.suptitle(
    "Fig. 10: Confusion Matrix.", fontweight="bold", horizontalalignment="right"
)
plt.show()

Các phần tử trên đường chéo biểu diễn các dự báo đúng và các phần tử ngoài đường chéo là các dự báo sai. Để biết thêm về ma trận nhầm lẫn, bạn đọc nên đọc [A Review on Evaluation Metrics for Data Classification Evaluations.](https://airccse.org/journal/ijdkp/vol5.html#mar)


## **3. Kết luận**

Trong bài này ta đã bàn về luyện kim mô phỏng như một trong các kỹ thuật tinh chỉnh siêu tham số và cũng giới thiệu các hàm mục tiêu trong học máy. Ở bài tiếp theo, ta sẽ xem cách áp dụng tinh chỉnh siêu tham số Bayes vào một chiến lược giao dịch bitcoin.

**Tài liệu tham khảo**

1. Ampatzis, Christos, and Dario Izzo. "Machine Learning Techniques for Approximation of Objective Functions in Trajectory Optimisation." Proceedings of the IJCAI-09 workshop on Artificial Intelligence in Space, 2009.
2. Botchkarev, Alexei. "Evaluating Performance of Regression Machine Learning Models Using Multiple Error Metrics in Azure Machine Learning Studio." 2018.
3. Hossin, Mohammad, and Md Nasir Sulaiman. "A Review on Evaluation Metrics for Data Classification Evaluations." *International Journal of Data Mining & Knowledge Management Process*, vol 5, no. 2, 2015.